In [1]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'


In [2]:
import shutil
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import hashlib
import os
import subprocess
from pathlib import Path

WORKDIR = Path(os.environ.get('SNU_WORKDIR', '/content/snu_ai'))
DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
LOCAL_DATA_ROOT = WORKDIR / 'snuaichallenge_data'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'snuaichallenge_data'

def has_test_data(root):
    return all([
        (root / 'test.csv').is_file(),
        (root / 'test').is_dir(),
        (root / 'sample_submission.csv').is_file(),
    ])

def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    return matches[0] if matches else None

WORKDIR.mkdir(parents=True, exist_ok=True)
if has_test_data(DRIVE_DATA_ROOT):
    DATA_ROOT = DRIVE_DATA_ROOT
else:
    DATA_ROOT = LOCAL_DATA_ROOT
    if not has_test_data(DATA_ROOT):
        test_bundle = find_drive_file('snu_colab_test_bundle.tar')
        if test_bundle is None:
            raise FileNotFoundError(
                "MyDrive에서 snu_colab_test_bundle.tar를 찾지 못했습니다. "
                "MyDrive/snu ai/에 업로드해 주세요."
            )
        print(f'테스트 데이터 압축 해제: {test_bundle}')
        subprocess.run(['tar', '-xf', str(test_bundle), '-C', str(WORKDIR)], check=True)
        DRIVE_ROOT = test_bundle.parent
assert has_test_data(DATA_ROOT), f'테스트 데이터가 없습니다: {DATA_ROOT}'

DATA_DIR = str(DATA_ROOT)
OUTPUT_DIR = str(DRIVE_ROOT / 'snu_qwen3vl_outputs')

# 평가(EXP-015 fold-0 EM 0.5269)에 쓴 것과 동일한 가중치인지 해시로 고정한다.
ADAPTER_SAFETENSORS_SHA256 = 'e5be86b0f0e7a07a96c0901d227a6ff6c1901d30895b972956d3e51f852f3f09'
FINAL_ADAPTER_DIR = Path(OUTPUT_DIR) / 'qwen3vl_v2_os_final'
assert (FINAL_ADAPTER_DIR / 'adapter_config.json').is_file(), FINAL_ADAPTER_DIR
weights_path = FINAL_ADAPTER_DIR / 'adapter_model.safetensors'
assert weights_path.is_file(), weights_path
weights_sha = hashlib.sha256(weights_path.read_bytes()).hexdigest()
assert weights_sha == ADAPTER_SAFETENSORS_SHA256, (
    f'adapter sha256 불일치 — 평가된 가중치가 아닙니다: {weights_sha}'
)
print(f'DATA_DIR: {DATA_DIR}')
print(f'adapter 확인: {FINAL_ADAPTER_DIR} (sha256 일치)')


테스트 데이터 압축 해제: /content/drive/MyDrive/snu ai/snu_colab_test_bundle.tar
DATA_DIR: /content/snu_ai/snuaichallenge_data
adapter 확인: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_final (sha256 일치)


In [5]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46',
    'qwen-vl-utils==0.0.14', 'peft', 'torchao>=0.16.0',
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', 'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46', 'qwen-vl-utils==0.0.14', 'peft', 'torchao>=0.16.0'], returncode=0)

In [6]:
import os, re, ast, random, time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
ARM_NAME = 'qwen3vl_v2_os'

test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
assert len(test) == 819, len(test)
assert list(sample_submission.columns) == ['Id', 'Answer'], sample_submission.columns
assert set(sample_submission['Id']) == set(test['Id'])
print(f'test rows: {len(test)}')


test rows: 819


In [7]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for i in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[i+1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [c for c in clauses if c]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses-1] + [' '.join(clauses[max_clauses-1:])]
    if not clauses:
        clauses = [sentence]
    return clauses

STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like "
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    "This caption describes a single continuous action or a brief moment. "
    "Focus on subtle visual cues such as camera movement direction, object position changes, "
    "hand/body motion, and lighting changes to infer the temporal order."
)

def get_prompt_message(row, image_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    clause_count = len(split_clauses(sentence))
    hint = STRUCTURED_HINT if clause_count >= 3 else CONTINUOUS_HINT

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({"type": "image", "image": img_path})
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        f"{hint}\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})
    return [{"role": "user", "content": content}]


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False


In [8]:
import gc

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다. Colab 런타임을 GPU로 설정한 뒤 다시 실행하세요.')
print(f'추론 GPU: {torch.cuda.get_device_name(0)}')

for variable_name in ('model', 'base_model'):
    globals().pop(variable_name, None)
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR))
model.eval()
print(f'Adapter loaded: {FINAL_ADAPTER_DIR}')

for name, param in model.named_parameters():
    if 'lora_B' in name:
        print(f"LoRA B 값 확인: {param.abs().mean().item()}")
        break


추론 GPU: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Adapter loaded: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_final
LoRA B 값 확인: 0.00621941639110446


In [9]:
processor.tokenizer.padding_side = 'left'  # 배치 생성을 위해 필수

BATCH_SIZE = 8

model.eval()
test_results = []
started = time.perf_counter()

for start_idx in tqdm(range(0, len(test), BATCH_SIZE)):
    batch_rows = test.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, f'{DATA_DIR}/test') for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        test_results.append({
            'Id': row['Id'],
            'pred': pred,
            'parse_ok': parse_ok,
            'raw_output': output_text,
        })

elapsed = time.perf_counter() - started
assert len(test_results) == len(test), (len(test_results), len(test))
assert [r['Id'] for r in test_results] == test['Id'].tolist(), 'test Id 순서가 달라졌습니다.'
assert all(sorted(r['pred']) == [1, 2, 3, 4] for r in test_results), '비정상 pred가 있습니다.'

n_parse_fail = sum(1 for r in test_results if not r['parse_ok'])
answer_counts = Counter(str(r['pred']) for r in test_results)
print(f'추론 완료: {len(test_results)}행, {elapsed:.1f}s ({elapsed * 1000 / len(test_results):.1f} ms/샘플)')
print(f'파싱 실패: {n_parse_fail}')
print('최빈 답 5개:', answer_counts.most_common(5))

submission_df = pd.DataFrame({
    'Id': [r['Id'] for r in test_results],
    'Answer': [str(r['pred']) for r in test_results],
})
assert list(submission_df.columns) == list(sample_submission.columns)
assert submission_df['Id'].tolist() == test['Id'].tolist()

SUBMISSION_DIR = Path(OUTPUT_DIR) / f'{ARM_NAME}_submission'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
submission_path = SUBMISSION_DIR / f'submission_{ARM_NAME}.csv'
raw_path = SUBMISSION_DIR / f'test_raw_outputs_{ARM_NAME}.csv'
submission_df.to_csv(submission_path, index=False)
pd.DataFrame(test_results).to_csv(raw_path, index=False, encoding='utf-8-sig')

saved = pd.read_csv(submission_path)
assert len(saved) == 819 and saved['Answer'].notna().all()
print(f'저장 완료: {submission_path}')
print(f'저장 완료: {raw_path}')

from google.colab import files
files.download(str(submission_path))


  0%|          | 0/103 [00:00<?, ?it/s]

추론 완료: 819행, 693.0s (846.1 ms/샘플)
파싱 실패: 0
최빈 답 5개: [('[1, 2, 3, 4]', 129), ('[1, 3, 4, 2]', 53), ('[2, 3, 4, 1]', 37), ('[1, 4, 3, 2]', 37), ('[4, 1, 2, 3]', 36)]
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_submission/submission_qwen3vl_v2_os.csv
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_submission/test_raw_outputs_qwen3vl_v2_os.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>